# 02 - Baseline Regression (Ridge)

Este notebook treina e avalia o modelo base de Regressão Ridge para prever a cinemática da mão (ângulo contínuo) a partir das features de EMG no domínio do tempo.

In [1]:
import sys
from pathlib import Path

# Adiciona o diretório src ao path do Python para importar nossos módulos
sys.path.append(str(Path("..").resolve() / "src"))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from emg_hls4ml_mvp.dataset import build_feature_dataset

## 1. Carregamento dos Dados
Utilizamos o pipeline já encapsulado em `build_feature_dataset`.

In [2]:
data_root = Path("../data")
subject = "Sub001"
recording_id = "Sub001_1_05_450_0"
target_column = "index_z"

print(f"Carregando dados para {subject} - {recording_id}...")
X, y, _ = build_feature_dataset(
    data_root=data_root,
    subject=subject,
    recording_id=recording_id,
    target_column=target_column,
)

print(f"X shape: {X.shape}, y shape: {y.shape}")

Carregando dados para Sub001 - Sub001_1_05_450_0...
X shape: (3500, 384), y shape: (3500,)


## 2. Preparação (Split e Normalização)
Para validar dados temporais, o ideal é **não embaralhar** (shuffle=False) para podermos visualizar como a predição acompanha o final do movimento de maneira cronológica real.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features normalizadas com sucesso.")

Features normalizadas com sucesso.


## 3. Treino e Avaliação

In [4]:
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

MSE:  288.4403
RMSE: 16.9835
MAE:  13.4641
R2:   0.8974


## 4. Visualização
A melhor forma de avaliar a regressão no EMG é analisando como as curvas se alinham.